# LegalBERT Fine-Tuning

Fine-tune a LegalBERT-style sequence classifier on the focused CUAD splits exported by notebook 01.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
dataset = load_dataset('json', data_files={
    'train': '/content/legaleagle_train.jsonl',
    'validation': '/content/legaleagle_valid.jsonl',
    'test': '/content/legaleagle_test.jsonl',
})
labels = sorted(set(dataset['train']['label']))
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
def tokenize(batch):
    encoded = tokenizer(batch['text'], truncation=True, padding='max_length', max_length=384)
    encoded['labels'] = [label2id[label] for label in batch['label']]
    return encoded

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset['train'].column_names)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        'precision_macro': precision_metric.compute(predictions=y_pred, references=y_true, average='macro')['precision'],
        'recall_macro': recall_metric.compute(predictions=y_pred, references=y_true, average='macro')['recall'],
        'f1_macro': f1_metric.compute(predictions=y_pred, references=y_true, average='macro')['f1'],
    }


In [ ]:
args = TrainingArguments(
    output_dir='/content/legaleagle-legalbert',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate(tokenized['test'])


In [ ]:
trainer.save_model('/content/legaleagle_fine_tuned_legalbert')
tokenizer.save_pretrained('/content/legaleagle_fine_tuned_legalbert')
!zip -r /content/legaleagle_fine_tuned_legalbert.zip /content/legaleagle_fine_tuned_legalbert
